# Attention & Transformers — Real Dataset: Sentiment Classification

**Goal:** Fine-tune a pre-trained BERT model for sentiment classification on the IMDb movie reviews dataset.  
Then train our from-scratch encoder for comparison.

**Dataset:** IMDb (50k movie reviews, binary sentiment) — available via HuggingFace datasets

**Tasks:**
1. Load and explore IMDb dataset
2. Tokenize and create data loaders
3. Fine-tune `bert-base-uncased` with a classification head
4. Train our from-scratch Transformer encoder for comparison
5. Evaluate and interpret attention patterns

**Prerequisites:** `01_from_scratch.ipynb` (or at minimum, understanding of the Transformer encoder)

---

In [ ]:
# Check dependencies
import subprocess
import sys

required = ['torch', 'transformers', 'datasets', 'numpy', 'matplotlib', 'sklearn']
missing = []
for pkg in required:
    try:
        __import__(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f'Installing missing packages: {missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
    
print('All dependencies available.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from transformers import BertTokenizerFast, BertModel, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 1. Load the IMDb Dataset

In [ ]:
# Load IMDb from HuggingFace Hub
dataset = load_dataset('imdb')
print(dataset)
print(f"\nTrain size: {len(dataset['train'])}")
print(f"Test size:  {len(dataset['test'])}")

In [ ]:
# Explore a few examples
for i in range(3):
    example = dataset['train'][i]
    label_str = 'positive' if example['label'] == 1 else 'negative'
    print(f'--- Example {i} ({label_str}) ---')
    print(example['text'][:300])
    print()

In [ ]:
# Check label distribution
train_labels = dataset['train']['label']
neg = sum(1 for l in train_labels if l == 0)
pos = sum(1 for l in train_labels if l == 1)
print(f'Negative: {neg} ({neg/len(train_labels):.1%})')
print(f'Positive: {pos} ({pos/len(train_labels):.1%})')

## 2. Tokenize and Create Data Loaders

In [ ]:
MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 256        # Truncate long reviews (most are <512 tokens)
BATCH_SIZE = 16

tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)
print(f'Vocab size: {tokenizer.vocab_size}')
print(f'Max model length: {tokenizer.model_max_length}')

In [ ]:
def tokenize_batch(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LEN,
        return_tensors=None   # return lists; DataLoader converts to tensors
    )

# Tokenize the full dataset
tokenized = dataset.map(tokenize_batch, batched=True, batch_size=1000)
tokenized = tokenized.with_format('torch')
print('Tokenization complete.')
print(f'Features: {list(tokenized["train"].features.keys())}')

In [ ]:
# Use a subset for faster training in this notebook
# Remove this for full training
N_TRAIN = 2000
N_TEST  = 500

train_subset = tokenized['train'].select(range(N_TRAIN))
test_subset  = tokenized['test'].select(range(N_TEST))

def collate_fn(batch):
    return {
        'input_ids':      torch.stack([b['input_ids'] for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'labels':         torch.tensor([b['label'] for b in batch]),
    }

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader  = DataLoader(test_subset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'Training batches: {len(train_loader)}')
print(f'Test batches: {len(test_loader)}')

## 3. Fine-tune BERT for Sentiment Classification

Using `bert-base-uncased` with a linear classification head on top of the `[CLS]` token.

In [ ]:
class BertClassifier(nn.Module):
    def __init__(self, num_labels=2, dropout=0.1):
        super().__init__()
        self.bert = BertModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
    
    def forward(self, input_ids, attention_mask):
        # BERT returns: last_hidden_state, pooler_output, ...
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # pooler_output = linear + tanh on [CLS] token
        cls_output = outputs.pooler_output  # (batch, hidden_size)
        return self.classifier(self.dropout(cls_output))  # (batch, num_labels)


bert_clf = BertClassifier().to(DEVICE)
n_params = sum(p.numel() for p in bert_clf.parameters())
print(f'Total parameters: {n_params:,}')
print(f'BERT hidden size: {bert_clf.bert.config.hidden_size}')

In [ ]:
EPOCHS = 2
LR = 2e-5

optimizer = optim.AdamW(bert_clf.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps
)
criterion = nn.CrossEntropyLoss()

print(f'Training for {EPOCHS} epochs, {total_steps} total steps')

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, n_correct, n_total = 0, 0, 0
    
    for i, batch in enumerate(loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        preds = logits.argmax(dim=-1)
        total_loss += loss.item()
        n_correct += (preds == labels).sum().item()
        n_total += labels.size(0)
        
        if (i + 1) % 20 == 0:
            print(f'  Step {i+1}/{len(loader)} | Loss: {loss.item():.4f} | Acc: {n_correct/n_total:.4f}')
    
    return total_loss / len(loader), n_correct / n_total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        total_loss += loss.item()
        
        preds = logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), acc, all_preds, all_labels

In [ ]:
train_losses, train_accs = [], []

for epoch in range(EPOCHS):
    print(f'\n=== Epoch {epoch+1}/{EPOCHS} ===')
    train_loss, train_acc = train_epoch(bert_clf, train_loader, optimizer, scheduler, criterion, DEVICE)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')

In [ ]:
# Evaluate on test set
test_loss, test_acc, test_preds, test_labels = evaluate(bert_clf, test_loader, criterion, DEVICE)
print(f'\nTest Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')
print('\nClassification Report:')
print(classification_report(test_labels, test_preds, target_names=['Negative', 'Positive']))

In [ ]:
# Plot training curve
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(train_losses, marker='o')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-entropy loss')

axes[1].plot(train_accs, marker='o', color='green')
axes[1].set_title('Training Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 4. Visualize Attention Patterns

Inspect what BERT is attending to for a sample review.

In [ ]:
def get_attention_weights(model, text, tokenizer, device, max_len=64):
    """Extract attention weights from BERT for a single text."""
    inputs = tokenizer(
        text, return_tensors='pt', truncation=True,
        max_length=max_len, padding='max_length'
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    model.eval()
    with torch.no_grad():
        outputs = model.bert(
            **inputs,
            output_attentions=True
        )
    
    # outputs.attentions: tuple of (batch, heads, seq_len, seq_len) per layer
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    # Find where padding starts (remove [PAD] tokens from view)
    pad_id = tokenizer.pad_token_id
    n_real = (inputs['input_ids'][0] != pad_id).sum().item()
    tokens = tokens[:n_real]
    
    return outputs.attentions, tokens


sample_text = "This movie was absolutely brilliant. The acting was superb and the story kept me engaged throughout."
attentions, tokens = get_attention_weights(bert_clf, sample_text, tokenizer, DEVICE)

print(f'Number of layers: {len(attentions)}')
print(f'Attention shape: {attentions[0].shape}')  # (1, 12, seq_len, seq_len)
print(f'Tokens: {tokens}')

In [ ]:
# Visualize attention heads from the last layer
layer_idx = -1  # last layer
n_real = len(tokens)

attn = attentions[layer_idx][0, :, :n_real, :n_real].cpu().numpy()  # (heads, n_real, n_real)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for head_idx, ax in enumerate(axes.flat):
    if head_idx >= attn.shape[0]:
        ax.axis('off')
        continue
    im = ax.imshow(attn[head_idx], cmap='Blues', vmin=0, vmax=attn[head_idx].max())
    ax.set_title(f'Head {head_idx}', fontsize=9)
    ax.set_xticks(range(n_real))
    ax.set_yticks(range(n_real))
    ax.set_xticklabels(tokens, rotation=90, fontsize=7)
    ax.set_yticklabels(tokens, fontsize=7)

plt.suptitle(f'BERT Attention Weights (Layer {12 + layer_idx + 1}/12)', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Quick Inference — Try Your Own Reviews

In [ ]:
def predict_sentiment(text, model, tokenizer, device, max_len=256):
    inputs = tokenizer(
        text, return_tensors='pt', truncation=True,
        max_length=max_len, padding='max_length'
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    model.eval()
    with torch.no_grad():
        logits = model(inputs['input_ids'], inputs['attention_mask'])
    
    probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    label = 'POSITIVE' if probs[1] > probs[0] else 'NEGATIVE'
    confidence = max(probs)
    return label, confidence, probs


test_reviews = [
    "Absolutely fantastic! One of the best films I've seen in years.",
    "Terrible. Boring, poorly acted, and a complete waste of time.",
    "It was okay. Some good moments but ultimately forgettable.",
]

for review in test_reviews:
    label, conf, probs = predict_sentiment(review, bert_clf, tokenizer, DEVICE)
    print(f'Review: "{review[:60]}..."' if len(review) > 60 else f'Review: "{review}"')
    print(f'  → {label} (confidence: {conf:.3f}  |  neg: {probs[0]:.3f}, pos: {probs[1]:.3f})')
    print()

## 6. Reflection & Key Takeaways

### What we did
- Loaded a real-world NLP dataset (IMDb) from HuggingFace
- Applied BERT tokenization (WordPiece, [CLS]/[SEP] tokens, padding)
- Fine-tuned `bert-base-uncased` with a classification head on [CLS] representation
- Visualized what each attention head learns

### What to notice

**Attention patterns:**
- Some heads focus on `[CLS]` — aggregating sequence information
- Some heads are positional (attending to neighbors)
- Some heads track syntactic relationships (subject ↔ verb)

**Fine-tuning efficiency:**
- BERT has 110M parameters but we only need a small classifier head
- With 2000 examples and 2 epochs, we still get strong performance — transfer learning!

### Experiments to try

1. **Full dataset**: remove `N_TRAIN` / `N_TEST` limits — expect >93% accuracy
2. **Freeze BERT, train only the head**: compare with full fine-tuning
3. **Different pooling**: use mean of all token representations instead of [CLS]
4. **Smaller models**: try `distilbert-base-uncased` — 40% smaller, similar performance
5. **Different tasks**: swap IMDb for SST-2, MNLI, or SQuAD

### Connecting to the architecture

BERT = 12 Transformer encoder blocks × (MHA + FFN + LayerNorm + residuals)  
Exactly what we built in `01_from_scratch.ipynb` — just 12 layers deep, with 768-dim embeddings and 12 heads.

### Next steps

- `03_llms_genai/language_modeling/` — decoder-only architecture, next-token prediction
- `03_llms_genai/finetuning/` — LoRA and parameter-efficient fine-tuning
- `frameworks/pytorch/` — training loop patterns, mixed precision, gradient accumulation